# 007 - JupyterLab Sidebar Chatbot 데모

JupyterLab **우측 사이드바**에 뜨는 챗봇입니다. 두뇌는 **langgraph 그래프(deepagents) + Claude** 이고, **LangGraph 생태계 네이티브 서빙(`langgraph dev`)** 으로 띄웁니다.

| 단계 | 하는 일 | jupyter 재시작? |
|---|---|---|
| ① 프론트 설치 | 셀에서 `%pip install <wheel>` → 브라우저 새로고침 → 우측 💬 탭 등장 | ❌ 불필요 |
| ② 두뇌 시작 | `langgraph dev` (LangGraph Server, localhost:2024) | ❌ 불필요 |
| ③ 대화 | 💬 탭에서 입력 → thread/run API → Claude(deepagents) 응답 | — |

> ⚠️ **온라인/개발 전용**: 두뇌가 Claude API(외부 네트워크)를 호출합니다. 키는 `ANTHROPIC_API_KEY` 환경변수로만 주입(노트북에 키 금지). **wheel 에는 의존성이 없으니** deepagents·langgraph-cli·모델 라이브러리는 직접 설치하세요.
>
> ⚠️ 전제: 브라우저의 `127.0.0.1` 과 이 커널/서버가 **같은 기계**여야 합니다(로컬/컨테이너/SSH 포트포워딩). single-file `.py` 가 아니라 빌드가 필요한 익스텐션입니다 — 자세한 건 `README.md` 참고.

## ① 프론트엔드 설치 — 노트북 셀에서 (jupyter 재시작 불필요)

터미널을 못 여는 환경이라면 **셀에서 직접 `%pip install`** 하면 됩니다. `%pip` 매직은 지금 이 커널(=jupyter 서버와 같은 env)에 설치하므로, labextension 자산이 `share/jupyter/labextensions/` 에 들어갑니다. 폐쇄망에서는 반입한 `.whl` 경로를 지정하세요.

In [ ]:
# 빌드 산출물 dist/ 의 wheel 을 설치 (`pip wheel . -w dist` 결과). 폐쇄망에선 반입한 .whl 경로로 바꾸세요.
%pip install dist/jlab_sidebar_chatbot-0.1.0-py3-none-any.whl
# 설치 후 → 브라우저에서 JupyterLab 페이지를 '새로고침'(Cmd/Ctrl+R) 하면 우측에 💬 탭이 생깁니다.
# (서버 재시작 불필요: jupyter 는 페이지를 그릴 때마다 labextensions 폴더를 다시 스캔하기 때문)

## ② LangGraph Server 시작 — `langgraph dev`

두뇌는 **LangGraph 생태계 네이티브 서빙**(`langgraph dev`)으로 띄웁니다. `langgraph.json` 이 `graph.py:make_graph` 를 가리키고, 우측 💬 탭은 이 서버의 thread/run API(`/threads`, `/runs/wait`)를 호출합니다. 멀티턴(thread 영속화)은 LangGraph 플랫폼이 제공합니다.

> 의존성은 폐쇄망에서 직접 설치하세요: `pip install deepagents "langgraph-cli[inmem]" langchain-anthropic` (모델 라이브러리는 선택).

In [ ]:
# LangGraph Server(langgraph dev)를 백그라운드로 시작. langgraph.json 이 있는 이 폴더에서 실행됩니다.
# ANTHROPIC_API_KEY 는 jupyter 서버 env 에서 커널이 상속합니다.
# (터미널에서 `langgraph dev --allow-blocking` 로 직접 실행하는 게 더 idiomatic 합니다.)
import subprocess
_lg = subprocess.Popen(
    ["langgraph", "dev", "--allow-blocking", "--no-browser", "--no-reload", "--port", "2024"]
)
print("LangGraph Server 시작 중 → http://127.0.0.1:2024  (잠시 후 우측 💬 탭에서 대화)")
# 중지:  _lg.terminate()

## ③ 모델·시스템 프롬프트 바꾸기

서빙되는 그래프는 `jlab_sidebar_chatbot/graph.py` 의 `make_graph()` 입니다(`langgraph.json` 이 이걸 가리킴). 바꾸려면 `make_graph()` 를 수정하세요:

- **모델**: `ANTHROPIC_MODEL` 환경변수, 또는 `make_graph()` 의 `ChatAnthropic(model=...)` 수정
- **사내/다른 LLM**: `make_graph()` 의 `ChatAnthropic` 를 해당 langchain 챗모델로 교체 (wheel 에 모델 라이브러리 의존성이 없으므로 원하는 걸 직접 설치)
- **시스템 프롬프트·도구**: `create_deep_agent(system_prompt=..., tools=[...])` 인자 수정

수정 후 `langgraph dev` 를 재시작하면 반영됩니다.

> ⚠️ 기본 모델은 Claude(외부 네트워크) — 온라인/개발 전용. 키는 `ANTHROPIC_API_KEY` 환경변수.

## ④ 정리 — 설치한 익스텐션 삭제

데모가 끝나면 아래 셀로 두뇌 서버를 멈추고 패키지(프론트 labextension 포함)를 제거합니다. **jupyter 재시작 없이** 브라우저 새로고침만 하면 우측 💬 탭이 사라집니다.

In [ ]:
# (정리) LangGraph 서버 중지 + 설치했던 익스텐션 패키지 제거.
try:
    _lg.terminate()                       # ② 에서 띄운 LangGraph Server 중지
except NameError:
    pass

%pip uninstall -y jlab_sidebar_chatbot    # 프론트 labextension 자산 포함 패키지 삭제
# → 브라우저를 '새로고침'하면 우측 💬 탭이 사라집니다 (jupyter 서버 재시작 불필요)